# 3.4 AND OR search and AO*: practice notebook

When an action can end in several states, a list of actions is not enough. AND OR trees describe the agent's choices and nature's outcomes, AND OR search builds a plan with if then branches, and AO* finds the cheapest one.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### AND OR search in the erratic vacuum world

In [ ]:
# AND OR search in the erratic vacuum world. A state is (where, A_dirty, B_dirty).
# Erratic Suck: on a dirty square it cleans it, and sometimes the other square too.
#               On a clean square it sometimes drops dirt.
def results(state, action):
    where, a, b = state
    if action == "Right":
        return [("B", a, b)]
    if action == "Left":
        return [("A", a, b)]
    if where == "A":
        if a == 1:
            return sorted(set([("A", 0, b), ("A", 0, 0)]))
        return [("A", 0, b), ("A", 1, b)]
    if b == 1:
        return sorted(set([("B", a, 0), ("B", 0, 0)]))
    return [("B", a, 0), ("B", a, 1)]

def is_goal(state):
    return state[1] == 0 and state[2] == 0

def or_search(state, path):
    if is_goal(state):
        return []
    if state in path:
        return None                       # a loop: give up on this branch
    for action in ["Suck", "Right", "Left"]:
        plan = and_search(results(state, action), [state] + path)
        if plan is not None:
            return [action, plan]
    return None

def and_search(states, path):
    plans = {}
    for s in states:                      # EVERY outcome needs a plan
        plan = or_search(s, path)
        if plan is None:
            return None
        plans[s] = plan
    return plans

print(or_search(("A", 1, 1), []))

### Cost of the best solution graph on the slide example

In [ ]:
# AO* style bottom up costs on the AND OR graph of the slides. Every edge costs 1.
# A node has a list of connectors. A connector is a list of children that must ALL be solved.
graph = {"A": [["B"], ["C", "D"]], "B": [["E"], ["F"]], "C": [["G"], ["H", "I"]], "D": [["J"]]}
h = {"E": 7, "F": 9, "G": 3, "H": 0, "I": 0, "J": 0}   # the leaves: their exact cost

def cost(node):
    if node not in graph:
        return h[node]
    best = None
    for connector in graph[node]:
        total = 0
        for child in connector:
            total = total + 1 + cost(child)          # edge cost 1 plus the child's cost
        print(node, "via", "+".join(connector), "costs", total)
        if best is None or total < best:
            best = total
    return best

print("cost of A:", cost("A"))

## Your turn

### Task 1: Cost of the best solution graph

Write `best_cost(graph, leaf, node)`. graph maps a node to a list of connectors (lists of children). leaf maps each leaf to its exact cost. Every edge costs 1. Return the cost of the cheapest solution graph below node: the min over connectors of the sum of (1 + cost of each child).

Hint: A leaf returns its own cost. Otherwise add up each connector and keep the smallest total.

In [ ]:
def best_cost(graph, leaf, node):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("best_cost({'A': [['B'], ['C', 'D']], 'B': [['E'], ['F']], 'C': [['G'], ['H', 'I']], 'D': [['J']]}, {'E': 7, 'F': 9, 'G': 3, 'H': 0, 'I': 0, 'J': 0}, 'A')", 5), ("best_cost({'A': [['B'], ['C', 'D']], 'B': [['E'], ['F']], 'C': [['G'], ['H', 'I']], 'D': [['J']]}, {'E': 7, 'F': 9, 'G': 3, 'H': 6, 'I': 0, 'J': 0}, 'A')", 7), ("best_cost({'A': [['B'], ['C', 'D']], 'B': [['E'], ['F']], 'C': [['G'], ['H', 'I']], 'D': [['J']]}, {'E': 7, 'F': 9, 'G': 3, 'H': 0, 'I': 0, 'J': 0}, 'B')", 8), ("best_cost({}, {'X': 4}, 'X')", 4)])

### Task 2: Does a conditional plan always reach the goal?

The erratic vacuum world is given by `results(state, action)` below. A plan is [] (done) or [action, {state: plan, ...}] with a sub plan for every outcome. Write `works(state, plan)` that returns True if following the plan from state always ends in a state with both squares clean, else False.

Hint: For an action, every outcome must appear in the dict and its sub plan must work.

In [ ]:
def results(state, action):
    where, a, b = state
    if action == "Right":
        return [("B", a, b)]
    if action == "Left":
        return [("A", a, b)]
    if where == "A":
        if a == 1:
            return sorted(set([("A", 0, b), ("A", 0, 0)]))
        return [("A", 0, b), ("A", 1, b)]
    if b == 1:
        return sorted(set([("B", a, 0), ("B", 0, 0)]))
    return [("B", a, 0), ("B", a, 1)]

def works(state, plan):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("works(('A', 1, 1), ['Suck', {('A', 0, 0): [], ('A', 0, 1): ['Right', {('B', 0, 1): ['Suck', {('B', 0, 0): []}]}]}])", True), ("works(('A', 1, 1), ['Suck', {('A', 0, 0): []}])", False), ("works(('A', 0, 0), [])", True), ("works(('A', 1, 0), [])", False)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Cost of the best solution graph

This is the true optimum. AO* reaches it while expanding only part of the graph, but only if h never overestimates. With h(H) = 6 the true best is 7 (A → C + D, C → G, D → J), while AO* with the slide estimate h(D) = 4 stops at 9.

In [ ]:
def best_cost(graph, leaf, node):
    if node not in graph:
        return leaf[node]
    best = None
    for connector in graph[node]:
        total = 0
        for child in connector:
            total = total + 1 + best_cost(graph, leaf, child)
        if best is None or total < best:
            best = total
    return best

In [ ]:
check([("best_cost({'A': [['B'], ['C', 'D']], 'B': [['E'], ['F']], 'C': [['G'], ['H', 'I']], 'D': [['J']]}, {'E': 7, 'F': 9, 'G': 3, 'H': 0, 'I': 0, 'J': 0}, 'A')", 5), ("best_cost({'A': [['B'], ['C', 'D']], 'B': [['E'], ['F']], 'C': [['G'], ['H', 'I']], 'D': [['J']]}, {'E': 7, 'F': 9, 'G': 3, 'H': 6, 'I': 0, 'J': 0}, 'A')", 7), ("best_cost({'A': [['B'], ['C', 'D']], 'B': [['E'], ['F']], 'C': [['G'], ['H', 'I']], 'D': [['J']]}, {'E': 7, 'F': 9, 'G': 3, 'H': 0, 'I': 0, 'J': 0}, 'B')", 8), ("best_cost({}, {'X': 4}, 'X')", 4)])

### Solution to task 2: Does a conditional plan always reach the goal?

The second test fails because the plan has no branch for state (A, 0, 1).

In [ ]:
def results(state, action):
    where, a, b = state
    if action == "Right":
        return [("B", a, b)]
    if action == "Left":
        return [("A", a, b)]
    if where == "A":
        if a == 1:
            return sorted(set([("A", 0, b), ("A", 0, 0)]))
        return [("A", 0, b), ("A", 1, b)]
    if b == 1:
        return sorted(set([("B", a, 0), ("B", 0, 0)]))
    return [("B", a, 0), ("B", a, 1)]

def works(state, plan):
    if len(plan) == 0:
        return state[1] == 0 and state[2] == 0
    action = plan[0]
    branches = plan[1]
    for s in results(state, action):
        if s not in branches:
            return False
        if not works(s, branches[s]):
            return False
    return True

In [ ]:
check([("works(('A', 1, 1), ['Suck', {('A', 0, 0): [], ('A', 0, 1): ['Right', {('B', 0, 1): ['Suck', {('B', 0, 0): []}]}]}])", True), ("works(('A', 1, 1), ['Suck', {('A', 0, 0): []}])", False), ("works(('A', 0, 0), [])", True), ("works(('A', 1, 0), [])", False)])